In [ ]:
import os
import json
import shutil
import subprocess
from dotenv import load_dotenv

from pathlib import Path
from openai import OpenAI, OpenAIError

# Load environment variables
load_dotenv()

In [ ]:
data_dir = "../data"

input_dir = os.path.join(data_dir, "1_parser")
input_attachments_dir = os.path.join(input_dir, "attachments")

output_dir = os.path.join(data_dir, "2_enrich", "a_audio")
tmp_dir = os.path.join(data_dir, "2_enrich", "transcribe_tmp")
os.makedirs(output_dir, exist_ok=True)
os.makedirs(tmp_dir, exist_ok=True)

In [ ]:
# Initialize OpenAI client with environment variables
api_key = os.getenv("MISTRAL_API_KEY_TRANSCRIBE")
base_url = os.getenv("MISTRAL_BASE_URL")

if not api_key:
    raise ValueError("MISTRAL_API_KEY_TRANSCRIBE environment variable is not set")
if not base_url:
    raise ValueError("MISTRAL_BASE_URL environment variable is not set")

client = OpenAI(api_key=api_key, base_url=base_url)

supported_extensions = {'.opus', '.mp3', '.mp4', '.mpeg', '.mpga', '.m4a', '.wav', '.webm'}

In [ ]:
def convert_to_mp3(src: Path, dst: Path) -> None:
    if not src.is_file():
        raise FileNotFoundError(f'Input file {src} not found.')

    cmd = ['ffmpeg', '-y', '-i', str(src), '-acodec', 'libmp3lame', str(dst)]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    except FileNotFoundError:
        raise RuntimeError('ffmpeg executable not found. Install ffmpeg and ensure it is on your PATH.')
    except subprocess.CalledProcessError as err:
        raise RuntimeError(f'ffmpeg failed: {err.stderr.decode(errors="ignore")}') from err

In [ ]:
def transcribe(mp3_path: Path) -> str:
    if not mp3_path.is_file():
        raise FileNotFoundError(f'MP3 file {mp3_path} not found.')

    with mp3_path.open('rb') as audio_file:
        try:
            result = client.audio.transcriptions.create(
                model='voxtral-mini-latest',
                file=audio_file,
                response_format='json',
                prompt="Die folgende Audio Datei enthält eine Sprachnachricht aus Whatsapp. Der Kontext ist Fehler auf einem Boot."
            )
        except OpenAIError as err:
            raise RuntimeError(f'OpenAI transcription failed: {err}') from err
    return result.text if hasattr(result, 'text') else str(result) # type: ignore

In [ ]:
all_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir)]
for idx, file_path in enumerate(all_files, 1):
    if not file_path.endswith('.json'):
        continue
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for msg in data.get('Messages', []):
        for attach in msg.get('Attachments', []):
            attachment_path = Path(input_attachments_dir, attach.get('path', ''))
            ext = attachment_path.suffix.lower()
            if ext in supported_extensions:
                if ext == '.opus':
                    print(f"Converting {attachment_path} to MP3")
                    mp3_path = Path(tmp_dir) / (Path(attach['path']).stem + '.mp3')
                    convert_to_mp3(attachment_path, mp3_path)
                    attachment_path = mp3_path
                transcript = transcribe(mp3_path)
                msg['text'] = msg.get('text', '') + '<audio-transcription-' + str(attach['id']) + '>' + transcript + '</audio-transcription-' + str(attach['id']) + '>'

    file_name = os.path.basename(file_path)
    output_path = os.path.join(output_dir, file_name)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Processed {file_path} ({idx}/{len(all_files)})")

In [ ]:
shutil.rmtree(tmp_dir, ignore_errors=True)